# ☕ Análise de Vendas de Café — Máquina Automática

Análise exploratória das vendas de uma máquina automática de café, com foco em receita, formas de pagamento, mix de produtos e padrões temporais.

- **Período:** 01/03/2024 a 23/03/2025
- **Localização:** Ucrânia (UTC+2)
- **Moeda:** Hryvnia ucraniana (UAH)

## Colunas do dataset tratado

| Coluna | Descrição |
|---|---|
| `date` / `datetime` | Data e horário da transação |
| `hour` | Horário no formato HH:MM |
| `valor_hryvnia` | Valor da venda em UAH |
| `coffee_name` | Produto vendido |
| `tipo_pagamento` | Cartão ou dinheiro |
| `dia_semana` / `mes_nome` / `ano` | Referências de calendário |
| `semana_ano` / `dia_mes` | Semana e dia do mês |
| `periodo_dia` | Manhã, tarde ou noite/madrugada |
| `feriado_nome` | Data especial cadastrada para análise |


In [ ]:
import pandas as pd
import numpy as np

# 1. CARREGAMENTO


In [ ]:
df_raw = pd.read_csv('index_1.csv')

colunas_obrigatorias = {'datetime', 'cash_type', 'card', 'money', 'coffee_name'}
colunas_ausentes = colunas_obrigatorias.difference(df_raw.columns)
if colunas_ausentes:
    raise KeyError(f"Colunas obrigatórias ausentes: {sorted(colunas_ausentes)}")

df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], errors='coerce')
df_raw['money'] = pd.to_numeric(df_raw['money'], errors='coerce')

if df_raw['datetime'].isna().any():
    raise ValueError("Há datas inválidas na coluna datetime.")
if df_raw['money'].isna().any():
    raise ValueError("Há valores de venda inválidos na coluna money.")

df_raw['date'] = df_raw['datetime'].dt.normalize()
df_raw = df_raw.loc[df_raw['date'] <= pd.Timestamp('2025-03-23')].copy()

duplicatas = int(df_raw.duplicated().sum())
nulos_card_dinheiro = int(df_raw.loc[df_raw['cash_type'].eq('cash'), 'card'].isna().sum())
nulos_card_outros = int(df_raw.loc[~df_raw['cash_type'].eq('cash'), 'card'].isna().sum())

print(f"Registros: {len(df_raw):,} | Colunas originais: {df_raw.shape[1]}")
print(f"Período: {df_raw['date'].min():%d/%m/%Y} a {df_raw['date'].max():%d/%m/%Y}")
print(f"Duplicatas: {duplicatas}")
print(f"IDs de cartão ausentes em vendas em dinheiro: {nulos_card_dinheiro}")
print(f"IDs de cartão ausentes nos demais pagamentos: {nulos_card_outros}")

# 2. ENGENHARIA DE FEATURES


In [ ]:
df_raw['hour'] = df_raw['datetime'].dt.strftime('%H:%M')
df_raw['hora_numerica'] = df_raw['datetime'].dt.hour
df_raw['dia_semana_num'] = df_raw['datetime'].dt.dayofweek
df_raw['mes_num'] = df_raw['datetime'].dt.month
df_raw['ano'] = df_raw['datetime'].dt.year
df_raw['semana_ano'] = df_raw['datetime'].dt.isocalendar().week.astype(int)
df_raw['dia_mes'] = df_raw['datetime'].dt.day

dias_map = {
    0: 'Segunda-feira', 1: 'Terça-feira', 2: 'Quarta-feira',
    3: 'Quinta-feira', 4: 'Sexta-feira', 5: 'Sábado', 6: 'Domingo'
}
meses_map = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}

df_raw['dia_semana'] = df_raw['dia_semana_num'].map(dias_map)
df_raw['mes_nome'] = df_raw['mes_num'].map(meses_map)
df_raw['mes_ano'] = df_raw['mes_nome'] + '/' + df_raw['ano'].astype(str)
df_raw['tipo_pagamento'] = df_raw['cash_type'].map({'card': 'Cartão', 'cash': 'Dinheiro'})
df_raw['valor_hryvnia'] = df_raw['money']

# Datas cadastradas para comparação; não interpretar automaticamente como feriados oficiais.
datas_especiais = {
    "2024-03-08": "Dia Internacional da Mulher",
    "2024-05-09": "Dia da Memória",
    "2024-06-23": "Pentecostes Ortodoxo",
    "2024-06-24": "Pentecostes Ortodoxo",
    "2024-06-28": "Dia da Constituição",
    "2025-01-07": "Natal Ortodoxo",
    "2025-01-08": "Natal Ortodoxo",
    "2025-03-08": "Dia Internacional da Mulher"
}
datas_especiais_dt = {
    pd.to_datetime(data).date(): nome for data, nome in datas_especiais.items()
}
df_raw['feriado_nome'] = df_raw['date'].dt.date.map(lambda data: datas_especiais_dt.get(data, '-'))

def classificar_periodo(hora):
    if 6 <= hora < 12:
        return 'Manhã (06h-11h)'
    if 12 <= hora < 18:
        return 'Tarde (12h-17h)'
    return 'Noite/madrugada (18h-05h)'

df_raw['periodo_dia'] = df_raw['hora_numerica'].apply(classificar_periodo)

# 3. EXPORTAÇÃO DE DADOS TRATADOS

In [ ]:
colunas_finais = [
    'date', 'datetime', 'coffee_name', 'hour', 'hora_numerica',
    'tipo_pagamento', 'valor_hryvnia', 'dia_semana_num', 'dia_semana',
    'mes_num', 'mes_nome', 'ano', 'semana_ano', 'dia_mes', 'mes_ano',
    'periodo_dia', 'feriado_nome'
]

colunas_ausentes = [coluna for coluna in colunas_finais if coluna not in df_raw.columns]
if colunas_ausentes:
    raise KeyError(f"Colunas esperadas não foram criadas: {colunas_ausentes}")

df_tratado = df_raw[colunas_finais].copy()
del df_raw

df_tratado.to_excel(
    'vendas_cafe_tratado.xlsx', index=False, sheet_name='Vendas', engine='openpyxl'
)
df_tratado.to_csv('vendas_cafe_tratado.csv', index=False, encoding='utf-8-sig')

print(f"Arquivos exportados: {len(df_tratado):,} registros")


# 4. KPIs GLOBAIS




In [ ]:
print("\n📈 ETAPA 4: INDICADORES-CHAVE DE PERFORMANCE (KPIs)")
print("-" * 80)

receita_total = df_tratado['valor_hryvnia'].sum()
total_transacoes = len(df_tratado)
ticket_medio = df_tratado['valor_hryvnia'].mean()
dias_operacao = df_tratado['date'].nunique()
vendas_cartao = len(df_tratado[df_tratado['tipo_pagamento'] == 'Cartão'])
vendas_dinheiro = len(df_tratado[df_tratado['tipo_pagamento'] == 'Dinheiro'])
vendas_feriados = len(df_tratado[df_tratado['feriado_nome'] != '-'])

print(f"Período analisado      : {df_tratado['date'].min().date()} → {df_tratado['date'].max().date()}")
print(f"Dias com operação      : {dias_operacao}")
print(f"Transações totais      : {total_transacoes:,}")
print(f"Receita total          : {receita_total:,.2f} UAH")
print(f"Ticket médio           : {ticket_medio:.2f} UAH")
print(f"Ticket mínimo          : {df_tratado['valor_hryvnia'].min():.2f} UAH")
print(f"Ticket máximo          : {df_tratado['valor_hryvnia'].max():.2f} UAH")
print(f"Desvio padrão          : {df_tratado['valor_hryvnia'].std():.2f} UAH")
print(f"Vendas média/dia       : {total_transacoes / dias_operacao:.2f}")
print(f"Receita média/dia      : {receita_total / dias_operacao:.2f} UAH")
print(f"Cartão                 : {vendas_cartao:,} ({vendas_cartao/total_transacoes*100:.1f}%)")
print(f"Dinheiro               : {vendas_dinheiro:,} ({vendas_dinheiro/total_transacoes*100:.1f}%)")
print(f"Vendas em feriados     : {vendas_feriados:,} ({vendas_feriados/total_transacoes*100:.1f}%)")

# 5. ANÁLISE POR MÉTODO DE PAGAMENTO

In [ ]:
print("\n💳 ETAPA 5: ANÁLISE POR MÉTODO DE PAGAMENTO")
print("-" * 80)

for metodo in ['Cartão', 'Dinheiro']:
    filtro = df_tratado[df_tratado['tipo_pagamento'] == metodo]
    qtd = len(filtro)
    receita = filtro['valor_hryvnia'].sum()
    ticket = filtro['valor_hryvnia'].mean()
    perc_qtd = (qtd / total_transacoes) * 100
    perc_receita = (receita / receita_total) * 100

    print(f"\n{metodo}:")
    print(f"  • Transações  : {qtd:>5} ({perc_qtd:>5.1f}% do volume)")
    print(f"  • Receita     : {receita:>10,.2f} UAH ({perc_receita:>5.1f}% do total)")
    print(f"  • Ticket médio: {ticket:>10.2f} UAH")

# 6. ANÁLISE POR DATAS ESPECIAIS CADASTRADAS

In [ ]:
df_tratado['eh_feriado'] = df_tratado['feriado_nome'].ne('-')

analise_feriados = df_tratado.groupby('eh_feriado').agg(
    Transacoes=('valor_hryvnia', 'count'),
    Receita=('valor_hryvnia', 'sum'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).rename(index={False: 'Dia sem data especial', True: 'Data especial'})

analise_feriados['Perc_Receita_%'] = (
    analise_feriados['Receita'] / receita_total * 100
).round(2)

print(analise_feriados.round(2).to_string())
print("\nA classificação depende das datas cadastradas; a amostra não demonstra causalidade.")

# 7. ANÁLISE ABC — MIX DE PRODUTOS

In [ ]:
abc = df_tratado.groupby('coffee_name').agg(
    Receita_Total=('valor_hryvnia', 'sum'),
    Quantidade=('valor_hryvnia', 'count'),
    Preco_Medio=('valor_hryvnia', 'mean')
).sort_values('Receita_Total', ascending=False)

abc['Receita_%'] = abc['Receita_Total'] / receita_total * 100
abc['Receita_Acumulada_%'] = abc['Receita_%'].cumsum()
abc['Classe_ABC'] = abc['Receita_Acumulada_%'].apply(
    lambda x: 'A' if x <= 80 else ('B' if x <= 95 else 'C')
)
abc = abc.round({'Receita_Total': 2, 'Preco_Medio': 2, 'Receita_%': 2, 'Receita_Acumulada_%': 2})

print(abc.to_string())

resumo_abc = abc.groupby('Classe_ABC', sort=True).agg(
    Produtos=('Quantidade', 'count'),
    Vendas=('Quantidade', 'sum'),
    Receita_Percentual=('Receita_%', 'sum')
)
print("\nResumo ABC:")
print(resumo_abc.round(1).to_string())

# 8. ANÁLISE MENSAL

In [ ]:
mensal = df_tratado.groupby(['ano', 'mes_num', 'mes_ano']).agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).reset_index().sort_values(['ano', 'mes_num']).reset_index(drop=True)

mensal['Perc_Total'] = mensal['Receita'] / receita_total * 100
mensal['Crescimento_%'] = mensal['Receita'].pct_change() * 100

ultima_data = df_tratado['date'].max()
mes_atual_parcial = ultima_data.day < ultima_data.days_in_month
mensal['Parcial'] = (
    mensal['ano'].eq(ultima_data.year)
    & mensal['mes_num'].eq(ultima_data.month)
    & mes_atual_parcial
)

# Não compara o mês parcial com o mês anterior completo.
mensal.loc[mensal['Parcial'], 'Crescimento_%'] = np.nan

print(f"{'Período':<15} {'Receita (UAH)':>14} {'Vendas':>8} {'Ticket':>10} {'% total':>9} {'Cresc.':>10}")
print('-' * 72)

for _, row in mensal.iterrows():
    crescimento = '—' if pd.isna(row['Crescimento_%']) else f"{row['Crescimento_%']:.1f}%"
    parcial = ' *' if row['Parcial'] else ''
    print(
        f"{row['mes_ano']:<15} {row['Receita']:>14,.2f} {int(row['Vendas']):>8} "
        f"{row['Ticket_Medio']:>10.2f} {row['Perc_Total']:>8.1f}% {crescimento:>10}{parcial}"
    )

print("\n* Mês parcial; crescimento não comparado com o mês anterior.")

# 9. ANÁLISE SEMANAL

In [ ]:
print("\n📆 ETAPA 9: ANÁLISE SEMANAL — SAZONALIDADE POR DIA DA SEMANA")
print("-" * 80)

ordem_dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira',
              'Sexta-feira', 'Sábado', 'Domingo']

semanal = df_tratado.groupby(pd.Categorical(df_tratado['dia_semana'], categories=ordem_dias, ordered=True)).agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).round(2)

print("\n" + f"{'Dia da Semana':<20} {'Vendas':>8} {'Receita':>15} {'Ticket':>12} {'%Total':>10}")
print("-" * 80)

for dia in ordem_dias:
    row = semanal.loc[dia]
    perc = (row['Receita'] / receita_total) * 100
    print(f"{dia:<20} {int(row['Vendas']):>8} {row['Receita']:>15,.2f} "
          f"{row['Ticket_Medio']:>12.2f} {perc:>9.1f}%")

melhor_dia = semanal['Receita'].idxmax()
pior_dia = semanal['Receita'].idxmin()
print(f"\n🏆 Melhor dia: {melhor_dia}")
print(f"⚠️  Pior dia:  {pior_dia}")

# 10. ANÁLISE HORÁRIA

In [ ]:
ordem_periodos = [
    'Manhã (06h-11h)',
    'Tarde (12h-17h)',
    'Noite/madrugada (18h-05h)'
]

por_periodo = df_tratado.groupby(
    pd.Categorical(df_tratado['periodo_dia'], categories=ordem_periodos, ordered=True),
    observed=True
).agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).round(2)

print(f"{'Período':<28} {'Vendas':>8} {'% vendas':>10} {'Receita (UAH)':>15} {'% receita':>10} {'Ticket':>10}")
print('-' * 85)

for periodo in ordem_periodos:
    if periodo not in por_periodo.index:
        continue
    row = por_periodo.loc[periodo]
    perc_vendas = row['Vendas'] / total_transacoes * 100
    perc_receita = row['Receita'] / receita_total * 100
    print(
        f"{periodo:<28} {int(row['Vendas']):>8} {perc_vendas:>9.1f}% "
        f"{row['Receita']:>15,.2f} {perc_receita:>9.1f}% {row['Ticket_Medio']:>10.2f}"
    )

por_hora = df_tratado.groupby('hora_numerica').agg(
    Receita=('valor_hryvnia', 'sum'),
    Vendas=('valor_hryvnia', 'count'),
    Ticket_Medio=('valor_hryvnia', 'mean')
).sort_values('Vendas', ascending=False).head(10)

print("\n10 horários com maior volume:")
print(f"{'Horário':>8} {'Vendas':>8} {'% total':>10} {'Receita (UAH)':>15} {'Ticket':>10}")
print('-' * 58)

for hora, row in por_hora.iterrows():
    perc = row['Vendas'] / total_transacoes * 100
    print(
        f"{int(hora):02d}:00 {int(row['Vendas']):>8} {perc:>9.1f}% "
        f"{row['Receita']:>15,.2f} {row['Ticket_Medio']:>10.2f}"
    )

## 11. Receita por dia de operação

Indicador complementar para comparar meses com quantidades diferentes de dias observados.

In [ ]:
diario = df_tratado.groupby('date').agg(
    vendas=('valor_hryvnia', 'size'),
    receita=('valor_hryvnia', 'sum')
).reset_index()

diario['mes'] = diario['date'].dt.to_period('M')
receita_mensal_diaria = diario.groupby('mes').agg(
    dias_com_vendas=('date', 'nunique'),
    receita_total=('receita', 'sum')
)
receita_mensal_diaria['receita_por_dia'] = (
    receita_mensal_diaria['receita_total'] / receita_mensal_diaria['dias_com_vendas']
).round(2)

print(receita_mensal_diaria.tail(5).to_string())